# 🏦 JPMC Consumer Banking - Hands-on Lab
## Notebook 03b: Data Transformation with Stored Procedures, Streams & Tasks (Imperative Approach)

---

### What You'll Learn
- Implementing the **same transformation logic** as Notebook 03 using stored procedures
- Using **Streams** for Change Data Capture (CDC) to process only new/changed rows
- Building a **Task DAG** with dependencies to orchestrate execution
- **Comparing** declarative (Dynamic Tables) vs imperative (Streams + SP + Tasks) approaches

### Why Compare?

Dynamic Tables are ideal for straightforward transformations where the logic is a single SELECT statement. But Streams + Stored Procedures + Tasks give you:
- **CDC (Change Data Capture)** - only process rows that changed (INSERTs, UPDATEs, DELETEs)
- **Conditional logic** (IF/ELSE, loops)
- **Multi-step operations** (INSERT then UPDATE then DELETE)
- **Error handling** (TRY/CATCH with custom logging)
- **Complex business rules** that don't fit in a single query

### Architecture: Streams + SP + Tasks
```
Source Tables ──► Streams (CDC) ──► Tasks ──► Stored Procedures ──► Target Tables

CUSTOMERS ──► STREAM_CUSTOMERS ──┐
                                 ├─► TASK_CUSTOMER_360 (SP) ──► CUSTOMER_360_SP
ACCOUNTS ───► STREAM_ACCOUNTS ───┘

TRANSACTIONS ► STREAM_TRANSACTIONS ──► TASK_TXN_SUMMARY (SP) ──► TXN_SUMMARY_SP

                                       TASK_EXEC_DASHBOARD (SP) ──► EXECUTIVE_DASHBOARD_SP
```

In [ ]:
-- Setup context
USE ROLE ACCOUNTADMIN;
USE DATABASE JPMC_CONSUMER_BANKING_HOL;
USE SCHEMA HOL_LAB;
USE WAREHOUSE HOL_MEDIUM_WH;

## Step 1: Create Target Tables for SP Output

Unlike Dynamic Tables (which auto-create their output table), stored procedures write into pre-existing tables.

In [ ]:
-- =============================================================================
-- TARGET TABLES (SP will write results here)
-- Using _SP suffix to differentiate from DT outputs
-- =============================================================================

CREATE OR REPLACE TABLE CUSTOMER_360_SP LIKE CUSTOMER_360;
CREATE OR REPLACE TABLE TXN_SUMMARY_SP LIKE TXN_SUMMARY;
CREATE OR REPLACE TABLE EXECUTIVE_DASHBOARD_SP LIKE EXECUTIVE_DASHBOARD;

-- Add audit columns to track SP execution
ALTER TABLE CUSTOMER_360_SP ADD COLUMN _REFRESHED_AT TIMESTAMP DEFAULT CURRENT_TIMESTAMP();
ALTER TABLE TXN_SUMMARY_SP ADD COLUMN _REFRESHED_AT TIMESTAMP DEFAULT CURRENT_TIMESTAMP();
ALTER TABLE EXECUTIVE_DASHBOARD_SP ADD COLUMN _REFRESHED_AT TIMESTAMP DEFAULT CURRENT_TIMESTAMP();

## Step 2: Create Streams for Change Data Capture (CDC)

Streams track DML changes (INSERT, UPDATE, DELETE) on source tables. When a stored procedure consumes a stream, it only processes **new/changed rows** since the last consumption - making the pipeline incremental.

**Key Stream Concepts:**
- A stream records the **delta** (changes) since last consumed
- Once a DML transaction reads the stream, the offset advances (stream is "consumed")
- `METADATA$ACTION` = INSERT or DELETE (UPDATEs appear as DELETE + INSERT)
- `METADATA$ISUPDATE` = TRUE if the row is part of an UPDATE
- `SYSTEM$STREAM_HAS_DATA()` checks if there are pending changes

In [ ]:
-- =============================================================================
-- STREAMS: Track changes on source tables for CDC
-- =============================================================================

-- Stream on CUSTOMERS table (captures new customers, profile updates)
CREATE OR REPLACE STREAM STREAM_CUSTOMERS
    ON TABLE CUSTOMERS
    APPEND_ONLY = FALSE
    COMMENT = 'CDC stream on CUSTOMERS - tracks inserts, updates, deletes';

-- Stream on ACCOUNTS table (captures new accounts, balance changes)
CREATE OR REPLACE STREAM STREAM_ACCOUNTS
    ON TABLE ACCOUNTS
    APPEND_ONLY = FALSE
    COMMENT = 'CDC stream on ACCOUNTS - tracks inserts, updates, deletes';

-- Stream on TRANSACTIONS table (append-only - transactions are never updated)
CREATE OR REPLACE STREAM STREAM_TRANSACTIONS
    ON TABLE TRANSACTIONS
    APPEND_ONLY = TRUE
    COMMENT = 'CDC stream on TRANSACTIONS - append-only, new transactions only';

-- Verify streams
SHOW STREAMS IN SCHEMA HOL_LAB;

In [ ]:
-- Check if streams have data (they should be empty initially until source tables change)
SELECT SYSTEM$STREAM_HAS_DATA('STREAM_CUSTOMERS') AS CUSTOMERS_HAS_CHANGES;
SELECT SYSTEM$STREAM_HAS_DATA('STREAM_ACCOUNTS') AS ACCOUNTS_HAS_CHANGES;
SELECT SYSTEM$STREAM_HAS_DATA('STREAM_TRANSACTIONS') AS TRANSACTIONS_HAS_CHANGES;

## Step 3: Simulate Changes to Populate Streams

Let's make some changes to the source tables so the streams have data to process.

In [ ]:
-- =============================================================================
-- SIMULATE CHANGES to populate streams with CDC data
-- =============================================================================

-- Insert new customers
INSERT INTO CUSTOMERS (CUSTOMER_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE, SSN_HASH, 
    DATE_OF_BIRTH, ADDRESS_JSON, SEGMENT, RISK_RATING, CREATED_AT, IS_ACTIVE)
VALUES 
    ('CUST_NEW_001', 'Alice', 'NewCustomer', 'alice.new@email.com', '+15551234567', 'hash001',
     '1990-05-15', '{"street":"100 New St","city":"New York","state":"NY","zip":"10001"}',
     'Premium', 'Low', CURRENT_TIMESTAMP()::VARCHAR, TRUE),
    ('CUST_NEW_002', 'Bob', 'RecentJoin', 'bob.recent@email.com', '+15559876543', 'hash002',
     '1985-11-20', '{"street":"200 Fresh Ave","city":"Chicago","state":"IL","zip":"60601"}',
     'Gold', 'Medium', CURRENT_TIMESTAMP()::VARCHAR, TRUE);

-- Update existing customer segments (simulates profile upgrades)
UPDATE CUSTOMERS SET SEGMENT = 'Premium', RISK_RATING = 'Low' 
WHERE CUSTOMER_ID IN ('CUST00000001', 'CUST00000002', 'CUST00000003');

-- Insert new transactions
INSERT INTO TRANSACTIONS (TXN_ID, ACCOUNT_ID, TXN_DATE, TXN_TYPE, AMOUNT, CURRENCY, 
    MERCHANT, CATEGORY, CHANNEL, STATUS, REFERENCE_NUM, DESCRIPTION)
VALUES 
    ('TXN_CDC_001', 'ACCT00000001', CURRENT_TIMESTAMP()::VARCHAR, 'DEBIT', 1500.00, 'USD',
     'Premium Store', 'SHOPPING', 'ONLINE', 'COMPLETED', 'REF_CDC_001', 'CDC Test Transaction 1'),
    ('TXN_CDC_002', 'ACCT00000002', CURRENT_TIMESTAMP()::VARCHAR, 'CREDIT', 5000.00, 'USD',
     'Wire Transfer', 'TRANSFER', 'ONLINE', 'COMPLETED', 'REF_CDC_002', 'CDC Test Transaction 2'),
    ('TXN_CDC_003', 'ACCT00000003', CURRENT_TIMESTAMP()::VARCHAR, 'DEBIT', 250.75, 'USD',
     'Grocery Mart', 'GROCERIES', 'POS', 'COMPLETED', 'REF_CDC_003', 'CDC Test Transaction 3');

-- Update account balances
UPDATE ACCOUNTS SET BALANCE = BALANCE + 5000 
WHERE ACCOUNT_ID = 'ACCT00000002';

SELECT 'Changes applied to source tables' AS STATUS;

In [ ]:
-- Now check the streams - they should have captured the changes!
SELECT 'STREAM_CUSTOMERS' AS STREAM_NAME, SYSTEM$STREAM_HAS_DATA('STREAM_CUSTOMERS') AS HAS_DATA
UNION ALL
SELECT 'STREAM_ACCOUNTS', SYSTEM$STREAM_HAS_DATA('STREAM_ACCOUNTS')
UNION ALL
SELECT 'STREAM_TRANSACTIONS', SYSTEM$STREAM_HAS_DATA('STREAM_TRANSACTIONS');

-- Preview what's in the CUSTOMERS stream (shows CDC metadata columns)
SELECT CUSTOMER_ID, FIRST_NAME, LAST_NAME, SEGMENT,
    METADATA$ACTION, METADATA$ISUPDATE, METADATA$ROW_ID
FROM STREAM_CUSTOMERS
LIMIT 10;

## Step 4: Create Stream-Based Stored Procedures (CDC / Incremental)

These SPs use MERGE to apply only the **changed rows** from streams - true incremental CDC processing. The key pattern:
1. Check if stream has data (`SYSTEM$STREAM_HAS_DATA`)
2. MERGE changes into target table (INSERT new, UPDATE modified, DELETE removed)
3. Once the SP transaction commits, the stream offset advances automatically

In [ ]:
-- =============================================================================
-- STORED PROCEDURE: SP_CDC_CUSTOMER_360
-- Uses STREAM_CUSTOMERS and STREAM_ACCOUNTS for incremental CDC processing
-- =============================================================================

CREATE OR REPLACE PROCEDURE SP_CDC_CUSTOMER_360()
RETURNS VARCHAR
LANGUAGE SQL
EXECUTE AS CALLER
AS
BEGIN
    LET start_time TIMESTAMP := CURRENT_TIMESTAMP();
    LET row_count INTEGER := 0;
    
    BEGIN
        -- Only process if there are changes in either stream
        IF (SYSTEM$STREAM_HAS_DATA('STREAM_CUSTOMERS') OR SYSTEM$STREAM_HAS_DATA('STREAM_ACCOUNTS')) THEN
            
            -- MERGE changed customers into CUSTOMER_360_SP
            -- This handles INSERTs (new customers), UPDATEs (profile changes), and DELETEs
            MERGE INTO CUSTOMER_360_SP tgt
            USING (
                SELECT
                    c.CUSTOMER_ID,
                    c.FIRST_NAME,
                    c.LAST_NAME,
                    c.EMAIL,
                    c.SEGMENT,
                    c.RISK_RATING,
                    c.CREATED_AT AS CUSTOMER_SINCE,
                    c.IS_ACTIVE,
                    COUNT(DISTINCT a.ACCOUNT_ID) AS TOTAL_ACCOUNTS,
                    SUM(CASE WHEN a.STATUS = 'ACTIVE' THEN 1 ELSE 0 END) AS ACTIVE_ACCOUNTS,
                    SUM(a.BALANCE) AS TOTAL_BALANCE,
                    AVG(a.BALANCE) AS AVG_ACCOUNT_BALANCE,
                    MAX(a.LAST_ACTIVITY_DATE) AS LAST_ACTIVITY,
                    SUM(CASE WHEN a.ACCOUNT_TYPE = 'CHECKING' THEN a.BALANCE ELSE 0 END) AS CHECKING_BALANCE,
                    SUM(CASE WHEN a.ACCOUNT_TYPE = 'SAVINGS' THEN a.BALANCE ELSE 0 END) AS SAVINGS_BALANCE,
                    SUM(CASE WHEN a.ACCOUNT_TYPE = 'MONEY_MARKET' THEN a.BALANCE ELSE 0 END) AS MONEY_MARKET_BALANCE,
                    PARSE_JSON(c.ADDRESS_JSON):city::VARCHAR AS CITY,
                    PARSE_JSON(c.ADDRESS_JSON):state::VARCHAR AS STATE
                FROM CUSTOMERS c
                LEFT JOIN ACCOUNTS a ON c.CUSTOMER_ID = a.CUSTOMER_ID
                WHERE c.CUSTOMER_ID IN (
                    SELECT CUSTOMER_ID FROM STREAM_CUSTOMERS
                    UNION
                    SELECT CUSTOMER_ID FROM STREAM_ACCOUNTS
                )
                GROUP BY ALL
            ) src
            ON tgt.CUSTOMER_ID = src.CUSTOMER_ID
            WHEN MATCHED THEN UPDATE SET
                tgt.FIRST_NAME = src.FIRST_NAME,
                tgt.LAST_NAME = src.LAST_NAME,
                tgt.EMAIL = src.EMAIL,
                tgt.SEGMENT = src.SEGMENT,
                tgt.RISK_RATING = src.RISK_RATING,
                tgt.IS_ACTIVE = src.IS_ACTIVE,
                tgt.TOTAL_ACCOUNTS = src.TOTAL_ACCOUNTS,
                tgt.ACTIVE_ACCOUNTS = src.ACTIVE_ACCOUNTS,
                tgt.TOTAL_BALANCE = src.TOTAL_BALANCE,
                tgt.AVG_ACCOUNT_BALANCE = src.AVG_ACCOUNT_BALANCE,
                tgt.LAST_ACTIVITY = src.LAST_ACTIVITY,
                tgt.CHECKING_BALANCE = src.CHECKING_BALANCE,
                tgt.SAVINGS_BALANCE = src.SAVINGS_BALANCE,
                tgt.MONEY_MARKET_BALANCE = src.MONEY_MARKET_BALANCE,
                tgt.CITY = src.CITY,
                tgt.STATE = src.STATE,
                tgt._REFRESHED_AT = CURRENT_TIMESTAMP()
            WHEN NOT MATCHED THEN INSERT (
                CUSTOMER_ID, FIRST_NAME, LAST_NAME, EMAIL, SEGMENT, RISK_RATING,
                CUSTOMER_SINCE, IS_ACTIVE, TOTAL_ACCOUNTS, ACTIVE_ACCOUNTS,
                TOTAL_BALANCE, AVG_ACCOUNT_BALANCE, LAST_ACTIVITY,
                CHECKING_BALANCE, SAVINGS_BALANCE, MONEY_MARKET_BALANCE,
                CITY, STATE, _REFRESHED_AT
            ) VALUES (
                src.CUSTOMER_ID, src.FIRST_NAME, src.LAST_NAME, src.EMAIL, src.SEGMENT, src.RISK_RATING,
                src.CUSTOMER_SINCE, src.IS_ACTIVE, src.TOTAL_ACCOUNTS, src.ACTIVE_ACCOUNTS,
                src.TOTAL_BALANCE, src.AVG_ACCOUNT_BALANCE, src.LAST_ACTIVITY,
                src.CHECKING_BALANCE, src.SAVINGS_BALANCE, src.MONEY_MARKET_BALANCE,
                src.CITY, src.STATE, CURRENT_TIMESTAMP()
            );
            
            row_count := SQLROWCOUNT;
            RETURN 'CDC SUCCESS: ' || :row_count || ' rows merged in ' || 
                   DATEDIFF('second', :start_time, CURRENT_TIMESTAMP()) || 's';
        ELSE
            RETURN 'SKIPPED: No changes detected in streams';
        END IF;
    EXCEPTION
        WHEN OTHER THEN
            RETURN 'ERROR: ' || SQLERRM;
    END;
END;

In [ ]:
-- =============================================================================
-- STORED PROCEDURE: SP_CDC_TXN_SUMMARY
-- Uses STREAM_TRANSACTIONS for incremental processing (append-only)
-- =============================================================================

CREATE OR REPLACE PROCEDURE SP_CDC_TXN_SUMMARY()
RETURNS VARCHAR
LANGUAGE SQL
EXECUTE AS CALLER
AS
BEGIN
    LET start_time TIMESTAMP := CURRENT_TIMESTAMP();
    LET row_count INTEGER := 0;
    
    BEGIN
        -- Only process if there are new transactions
        IF (SYSTEM$STREAM_HAS_DATA('STREAM_TRANSACTIONS')) THEN
            
            -- For append-only streams, we INSERT aggregated new transactions
            -- Use MERGE to update existing day/category/channel combinations
            MERGE INTO TXN_SUMMARY_SP tgt
            USING (
                SELECT
                    ACCOUNT_ID,
                    DATE_TRUNC('DAY', TO_TIMESTAMP(TXN_DATE)) AS TXN_DAY,
                    DATE_TRUNC('MONTH', TO_TIMESTAMP(TXN_DATE)) AS TXN_MONTH,
                    CATEGORY, CHANNEL, TXN_TYPE,
                    COUNT(*) AS TXN_COUNT,
                    SUM(AMOUNT) AS TOTAL_AMOUNT,
                    AVG(AMOUNT) AS AVG_AMOUNT,
                    MAX(AMOUNT) AS MAX_AMOUNT,
                    MIN(AMOUNT) AS MIN_AMOUNT,
                    SUM(CASE WHEN STATUS = 'COMPLETED' THEN 1 ELSE 0 END) AS COMPLETED_COUNT,
                    SUM(CASE WHEN STATUS = 'FAILED' THEN 1 ELSE 0 END) AS FAILED_COUNT,
                    SUM(CASE WHEN STATUS = 'REVERSED' THEN 1 ELSE 0 END) AS REVERSED_COUNT,
                    SUM(CASE WHEN AMOUNT > 1000 THEN 1 ELSE 0 END) AS HIGH_VALUE_TXN_COUNT
                FROM STREAM_TRANSACTIONS
                GROUP BY ALL
            ) src
            ON  tgt.ACCOUNT_ID = src.ACCOUNT_ID
                AND tgt.TXN_DAY = src.TXN_DAY
                AND tgt.CATEGORY = src.CATEGORY
                AND tgt.CHANNEL = src.CHANNEL
                AND tgt.TXN_TYPE = src.TXN_TYPE
            WHEN MATCHED THEN UPDATE SET
                tgt.TXN_COUNT = tgt.TXN_COUNT + src.TXN_COUNT,
                tgt.TOTAL_AMOUNT = tgt.TOTAL_AMOUNT + src.TOTAL_AMOUNT,
                tgt.AVG_AMOUNT = (tgt.TOTAL_AMOUNT + src.TOTAL_AMOUNT) / (tgt.TXN_COUNT + src.TXN_COUNT),
                tgt.MAX_AMOUNT = GREATEST(tgt.MAX_AMOUNT, src.MAX_AMOUNT),
                tgt.MIN_AMOUNT = LEAST(tgt.MIN_AMOUNT, src.MIN_AMOUNT),
                tgt.COMPLETED_COUNT = tgt.COMPLETED_COUNT + src.COMPLETED_COUNT,
                tgt.FAILED_COUNT = tgt.FAILED_COUNT + src.FAILED_COUNT,
                tgt.REVERSED_COUNT = tgt.REVERSED_COUNT + src.REVERSED_COUNT,
                tgt.HIGH_VALUE_TXN_COUNT = tgt.HIGH_VALUE_TXN_COUNT + src.HIGH_VALUE_TXN_COUNT,
                tgt._REFRESHED_AT = CURRENT_TIMESTAMP()
            WHEN NOT MATCHED THEN INSERT (
                ACCOUNT_ID, TXN_DAY, TXN_MONTH, CATEGORY, CHANNEL, TXN_TYPE,
                TXN_COUNT, TOTAL_AMOUNT, AVG_AMOUNT, MAX_AMOUNT, MIN_AMOUNT,
                COMPLETED_COUNT, FAILED_COUNT, REVERSED_COUNT, HIGH_VALUE_TXN_COUNT,
                _REFRESHED_AT
            ) VALUES (
                src.ACCOUNT_ID, src.TXN_DAY, src.TXN_MONTH, src.CATEGORY, src.CHANNEL, src.TXN_TYPE,
                src.TXN_COUNT, src.TOTAL_AMOUNT, src.AVG_AMOUNT, src.MAX_AMOUNT, src.MIN_AMOUNT,
                src.COMPLETED_COUNT, src.FAILED_COUNT, src.REVERSED_COUNT, src.HIGH_VALUE_TXN_COUNT,
                CURRENT_TIMESTAMP()
            );
            
            row_count := SQLROWCOUNT;
            RETURN 'CDC SUCCESS: ' || :row_count || ' rows merged in ' || 
                   DATEDIFF('second', :start_time, CURRENT_TIMESTAMP()) || 's';
        ELSE
            RETURN 'SKIPPED: No new transactions in stream';
        END IF;
    EXCEPTION
        WHEN OTHER THEN
            RETURN 'ERROR: ' || SQLERRM;
    END;
END;

In [ ]:
-- =============================================================================
-- STORED PROCEDURE: SP_REFRESH_EXECUTIVE_DASHBOARD (Python-based)
-- (Depends on CUSTOMER_360_SP and TXN_SUMMARY_SP being refreshed first)
-- =============================================================================

CREATE OR REPLACE PROCEDURE SP_REFRESH_EXECUTIVE_DASHBOARD()
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'run'
EXECUTE AS CALLER
AS
$$
import time
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import *

def run(session):
    start_time = time.time()
    
    try:
        # Truncate target table
        session.sql("TRUNCATE TABLE EXECUTIVE_DASHBOARD_SP").collect()
        
        # Load source tables
        c360 = session.table("CUSTOMER_360_SP")
        accounts = session.table("ACCOUNTS")
        txn_summary = session.table("TXN_SUMMARY_SP")
        
        # Join Customer 360 with Accounts and Transaction Summary
        joined_df = c360.join(accounts, c360["CUSTOMER_ID"] == accounts["CUSTOMER_ID"], "left") \
            .join(txn_summary, accounts["ACCOUNT_ID"] == txn_summary["ACCOUNT_ID"], "left") \
            .filter(F.col("TXN_MONTH") == F.date_trunc("MONTH", F.current_date()))
        
        # Aggregate to build executive dashboard
        dashboard_df = joined_df.group_by(
            c360["STATE"], c360["SEGMENT"]
        ).agg(
            F.count_distinct(c360["CUSTOMER_ID"]).alias("TOTAL_CUSTOMERS"),
            F.sum(F.when(c360["IS_ACTIVE"] == True, F.lit(1)).otherwise(F.lit(0))).alias("ACTIVE_CUSTOMERS"),
            F.avg(c360["TOTAL_BALANCE"]).alias("AVG_CUSTOMER_BALANCE"),
            F.sum(c360["TOTAL_BALANCE"]).alias("TOTAL_DEPOSITS"),
            F.sum(F.col("TXN_COUNT")).alias("MONTHLY_TXN_COUNT"),
            F.sum(F.col("TOTAL_AMOUNT")).alias("MONTHLY_TXN_VOLUME"),
            F.avg(F.col("AVG_AMOUNT")).alias("AVG_TXN_SIZE"),
            F.sum(F.col("FAILED_COUNT")).alias("MONTHLY_FAILED_TXNS"),
            F.round(
                F.sum(F.col("FAILED_COUNT")) / F.coalesce(F.sum(F.col("TXN_COUNT")), F.lit(1)) * F.lit(100), 
                F.lit(3)
            ).alias("FAILURE_RATE_PCT"),
            F.sum(F.col("HIGH_VALUE_TXN_COUNT")).alias("HIGH_VALUE_TXNS")
        ).with_column("_REFRESHED_AT", F.current_timestamp())
        
        # Write results
        dashboard_df.write.mode("append").save_as_table("EXECUTIVE_DASHBOARD_SP")
        
        row_count = session.table("EXECUTIVE_DASHBOARD_SP").count()
        duration = round(time.time() - start_time, 1)
        
        return f"SUCCESS: {row_count} rows in {duration}s (Python SP)"
        
    except Exception as e:
        return f"ERROR: {str(e)}"
$$;

## Step 5: Test the CDC Stored Procedures

In [ ]:
-- Test each CDC procedure manually (should process the changes we made in Step 3)
CALL SP_CDC_CUSTOMER_360();
CALL SP_CDC_TXN_SUMMARY();
CALL SP_REFRESH_EXECUTIVE_DASHBOARD();

## Step 6: Create Task DAG with Stream-Triggered Execution

Tasks can use `WHEN SYSTEM$STREAM_HAS_DATA(...)` to **only run when there are actual changes**. This prevents unnecessary compute when nothing has changed.

In [ ]:
-- =============================================================================
-- TASK DAG with STREAM-BASED TRIGGERS (only runs when data changes)
-- =============================================================================

-- Root task: runs every 5 minutes, but ONLY executes if streams have data
CREATE OR REPLACE TASK TASK_ROOT
    WAREHOUSE = HOL_MEDIUM_WH
    SCHEDULE = 'USING CRON */5 * * * * America/New_York'
    COMMENT = 'Root task - checks streams every 5 min, only fires if changes exist'
    WHEN SYSTEM$STREAM_HAS_DATA('STREAM_CUSTOMERS') 
      OR SYSTEM$STREAM_HAS_DATA('STREAM_ACCOUNTS')
      OR SYSTEM$STREAM_HAS_DATA('STREAM_TRANSACTIONS')
AS
    SELECT 1;  -- No-op, just triggers children when streams have data

-- Level 1: CDC tasks (run in parallel after root)
CREATE OR REPLACE TASK TASK_CUSTOMER_360
    WAREHOUSE = HOL_MEDIUM_WH
    AFTER TASK_ROOT
    COMMENT = 'CDC: Merge customer/account changes into Customer 360'
AS
    CALL SP_CDC_CUSTOMER_360();

CREATE OR REPLACE TASK TASK_TXN_SUMMARY
    WAREHOUSE = HOL_MEDIUM_WH
    AFTER TASK_ROOT
    COMMENT = 'CDC: Merge new transactions into summary'
AS
    CALL SP_CDC_TXN_SUMMARY();

-- Level 2: Dependent task (runs after upstream CDC tasks complete)
CREATE OR REPLACE TASK TASK_EXEC_DASHBOARD
    WAREHOUSE = HOL_MEDIUM_WH
    AFTER TASK_CUSTOMER_360, TASK_TXN_SUMMARY
    COMMENT = 'Refresh Executive Dashboard after CDC processing'
AS
    CALL SP_REFRESH_EXECUTIVE_DASHBOARD();

In [ ]:
-- =============================================================================
-- ENABLE THE TASK DAG (resume from root - cascades to children)
-- =============================================================================

-- Resume all tasks (bottom-up order required)
ALTER TASK TASK_EXEC_DASHBOARD RESUME;
ALTER TASK TASK_CUSTOMER_360 RESUME;
ALTER TASK TASK_TXN_SUMMARY RESUME;
ALTER TASK TASK_ROOT RESUME;

-- Verify task DAG
SHOW TASKS IN SCHEMA HOL_LAB;

In [ ]:
-- Manually execute the root task to test the full DAG
EXECUTE TASK TASK_ROOT;

## Step 7: Monitor Task Execution

In [ ]:
-- =============================================================================
-- TASK MONITORING
-- =============================================================================

-- View task execution history
SELECT NAME, STATE, SCHEDULED_TIME, COMPLETED_TIME, RETURN_VALUE,
    DATEDIFF('second', SCHEDULED_TIME, COMPLETED_TIME) AS DURATION_SECONDS
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
    SCHEDULED_TIME_RANGE_START => DATEADD(HOUR, -2, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 20
))
ORDER BY SCHEDULED_TIME DESC;

-- View task dependencies
SELECT * FROM TABLE(INFORMATION_SCHEMA.TASK_DEPENDENTS(
    TASK_NAME => 'TASK_ROOT',
    RECURSIVE => TRUE
));

## Step 8: Side-by-Side Comparison

| Aspect | Dynamic Tables (NB 03) | Streams + SP + Tasks (NB 03b) |
|--------|----------------------|---------------------|
| **CDC mechanism** | Built-in (automatic) | Streams (explicit) |
| **Processing** | Declarative SELECT | MERGE with stream data |
| **Trigger** | TARGET_LAG (time-based) | `WHEN SYSTEM$STREAM_HAS_DATA()` |
| **Orchestration** | Automatic DAG | Manual AFTER dependencies |
| **Incremental** | Automatic | Stream-driven MERGE |
| **Error handling** | Automatic retry | Custom TRY/CATCH blocks |
| **Cost** | Runs only when data changes | Runs only when stream has data |
| **Conditional logic** | Not supported | Full IF/ELSE, loops, multi-step |
| **Best for** | Standard transformations | Complex CDC, multi-step ETL |

### Recommendation
- **Use Dynamic Tables** when your transformation is a single SELECT (90% of cases)
- **Use Streams + SP + Tasks** when you need CDC with custom MERGE logic, conditional processing, or multi-step operations
- **Mix both** - use DTs for simple aggregations, Streams + SPs for complex CDC workflows

In [ ]:
-- =============================================================================
-- CLEANUP: Suspend tasks after demo (to avoid ongoing charges)
-- =============================================================================

ALTER TASK TASK_ROOT SUSPEND;
-- Child tasks automatically suspend when root is suspended

-- Verify
SELECT NAME, STATE FROM TABLE(INFORMATION_SCHEMA.TASK_DEPENDENTS(
    TASK_NAME => 'TASK_ROOT', RECURSIVE => TRUE
));